# Working with APIs in Python
## MS Health Informatics – Application Development
### UT Southwestern Medical Center

---

## What is a REST API?

An **API** (Application Programming Interface) is a way for two software systems to communicate with each other. You can think of it as a waiter in a restaurant: you (the client) tell the waiter (the API) what you want, and the waiter goes to the kitchen (the server) and comes back with your order (the data).

**REST** stands for **Representational State Transfer**. It is an architectural style that uses the HTTP protocol (the same protocol your web browser uses) to exchange data. Let's break down each word:

| Term | Meaning |
|------|---------|
| **Representational** | Data is returned as a *representation* of a resource (commonly JSON or XML) |
| **State Transfer** | The client and server exchange *state* — the current snapshot of a resource |

### Resource URIs

A **URI** (Uniform Resource Identifier) uniquely identifies a resource on a server. In REST, every resource (a patient, a joke, a medication) has its own URI. For example:

```
https://r4.smarthealthit.org/Patient/08bb5ad8-79b2-47ef-a0ad-81ef0b4ab2c3
                ▲                     ▲              ▲
           Base server URL       Resource type     Resource ID
```

### HTTP Methods

REST APIs use standard HTTP *methods* (also called *verbs*) to describe the action being performed:

| Method   | Action                          | Analogy            |
|----------|---------------------------------|--------------------|
| `GET`    | Retrieve a resource             | Reading a file     |
| `POST`   | Send data to update a resource  | Editing a form     |
| `PUT`    | Create or fully replace a resource | Replacing a file |
| `DELETE` | Remove a resource               | Deleting a file    |

---

## Setup

We will use Python's `requests` library to make HTTP calls. Run the cell below to install it if it is not already available.

In [ ]:
# Install the requests library if needed
%pip install requests --quiet

In [ ]:
import requests
import json

print("Libraries loaded successfully!")

---

## Part 1 – Your First GET Request

The `GET` method asks the server to *retrieve* a resource and return it to us. It is the most common HTTP method — every time you open a web page in your browser, you are sending a `GET` request.

We will start with a fun example: retrieving a random joke from the **Humor API**.

**Resource URI:** `https://api.humorapi.com/jokes/random`

The server will respond with a JSON object containing a joke.

In [ ]:
# ── GET request ────────────────────────────────────────────────────────────────
# We send a GET request to the joke API's resource URI.

url = "https://api.humorapi.com/jokes/random"
params = {"api-key": "DEMO"}  # Use your own API key from https://humorapi.com

response = requests.get(url, params=params)

# Every HTTP response includes a status code:
#   200 = OK (success)   404 = Not Found   500 = Server Error
print(f"Status code: {response.status_code}")
print(f"Response body:\n{response.text}")

### Making Multiple Requests

APIs are designed to be called repeatedly. Let's fetch **five different jokes** by looping over the same endpoint. Notice that each call returns a *different* random joke — the server generates a fresh representation each time.

In [ ]:
url = "https://api.humorapi.com/jokes/random"
params = {"api-key": "DEMO"}

print("Here are 5 random jokes:\n")
for i in range(1, 6):
    resp = requests.get(url, params=params)
    if resp.status_code == 200:
        data = resp.json()          # Parse the JSON response into a Python dict
        print(f"Joke #{i}: {data.get('joke', data)}")
    else:
        print(f"Request #{i} failed with status {resp.status_code}")
    print()

---

## Part 2 – Retrieving a FHIR Patient Record

Now let's apply what we learned to a real healthcare use case: **FHIR** (Fast Healthcare Interoperability Resources).

**FHIR** is a standard published by HL7 that defines how healthcare information should be structured and exchanged. Every type of healthcare data — patients, medications, lab results, appointments — is a **resource** with its own URI.

We will query the publicly accessible **SMART Health IT FHIR R4 sandbox server** to retrieve a patient record.

**Resource URI:** `https://r4.smarthealthit.org/Patient/08bb5ad8-79b2-47ef-a0ad-81ef0b4ab2c3`

By sending a `GET` request to this URI, we ask the server for the *current representation* of this specific patient resource.

In [ ]:
patient_id = "08bb5ad8-79b2-47ef-a0ad-81ef0b4ab2c3"
fhir_base = "https://r4.smarthealthit.org"
patient_url = f"{fhir_base}/Patient/{patient_id}"

# Ask for JSON format via the Accept header (FHIR also supports XML)
headers = {"Accept": "application/fhir+json"}

response = requests.get(patient_url, headers=headers)
print(f"Status code: {response.status_code}")

### What is JSON?

**JSON** (JavaScript Object Notation) is the most common data format returned by REST APIs. It is a lightweight, human-readable text format that represents structured data as *key-value pairs*, arrays, and nested objects.

Example of a small JSON object:
```json
{
  "resourceType": "Patient",
  "id": "08bb5ad8-79b2-47ef-a0ad-81ef0b4ab2c3",
  "name": [
    { "family": "Smith", "given": ["John"] }
  ]
}
```

JSON connects directly to the **RE** in REST: the server does not send you the actual patient — it sends you a **textual representation** of the patient resource in JSON format. The same resource could also be represented as XML, HTML, or any other format.

The cell below pretty-prints the full JSON response from the FHIR server so you can see how a real Patient resource looks.

In [ ]:
if response.status_code == 200:
    patient = response.json()   # Convert JSON text → Python dictionary
    print(json.dumps(patient, indent=2))
else:
    print(f"Could not retrieve patient. Status: {response.status_code}")
    print(response.text)

In [ ]:
# Extract some key fields to demonstrate working with JSON in Python
if response.status_code == 200:
    resource_type = patient.get("resourceType")
    patient_id_returned = patient.get("id")

    # Name is a list; grab the first entry
    name_entry = patient.get("name", [{}])[0]
    family = name_entry.get("family", "Unknown")
    given = " ".join(name_entry.get("given", []))

    birthdate = patient.get("birthDate", "Unknown")
    gender = patient.get("gender", "Unknown")

    print(f"Resource Type : {resource_type}")
    print(f"Patient ID    : {patient_id_returned}")
    print(f"Name          : {given} {family}")
    print(f"Date of Birth : {birthdate}")
    print(f"Gender        : {gender}")

---

## Part 3 – State Transfer: Modifying Resources

So far we have only *read* data (the **R** of CRUD). REST also supports creating, updating, and deleting resources — the **ST** (State Transfer) part of REST. This is how applications change the *state* stored on a server.

We will use the same SMART Health IT sandbox, which allows write operations for learning purposes.

### 3a. POST – Modify an existing resource

A `POST` request sends a payload (body) to a resource URI, asking the server to process it. POST semantics vary by API:

- In **generic REST**, `POST /collection` usually *creates* a new resource.
- In **FHIR**, `POST /Patient` creates a new patient with a server-assigned ID, while `POST /Patient/<id>` (used below) sends an update operation to a specific patient.

> **Real-world FHIR note:** Most FHIR servers use `PUT /Patient/<id>` for updates and `POST /Patient` for creation. The sandbox used here also accepts `POST` to a specific resource URI for updates, which lets us practice both verbs.

Below we take the patient record we retrieved, modify the phone number, and `POST` it back to the same URI to demonstrate that the resource *state* on the server has changed.

In [ ]:
# ── POST – update a FHIR Patient resource ─────────────────────────────────────
# We take the patient record we retrieved and modify a field,
# then POST it back to the server.

if response.status_code == 200:
    modified_patient = dict(patient)   # shallow copy so we don't mutate the original

    # Add / overwrite the telecom (phone/email) list
    modified_patient["telecom"] = [
        {"system": "phone", "value": "555-867-5309", "use": "home"}
    ]

    post_headers = {
        "Content-Type": "application/fhir+json",
        "Accept": "application/fhir+json",
    }

    # POST to /Patient/<id> to update the specific patient
    post_response = requests.post(
        patient_url,
        headers=post_headers,
        json=modified_patient,
    )

    print(f"POST status code : {post_response.status_code}")
    print(f"Response body    : {post_response.text[:500]}")
else:
    print("Skipping POST because the earlier GET did not succeed.")

### 3b. PUT – Create a new resource with a client-assigned ID

A `PUT` request *creates or fully replaces* a resource at a specific URI. Unlike `POST`, `PUT` is **idempotent** — calling it multiple times with the same data produces the same result (no duplicates).

> **FHIR convention:** `PUT /Patient/<id>` is the standard way to *update* a specific patient or to *create* one when the client chooses the ID. In production systems you will most often use `POST /Patient` to create (server assigns the ID) and `PUT /Patient/<id>` to update.

Here we create a brand-new patient by `PUT`-ting a complete Patient resource to a new URI with a UUID we generate ourselves. The server stores it under the ID we specify.

In [ ]:
# ── PUT – create a new Patient resource ───────────────────────────────────────
import uuid

new_patient_id = str(uuid.uuid4())   # Generate a random UUID for our new patient
new_patient_url = f"{fhir_base}/Patient/{new_patient_id}"

new_patient = {
    "resourceType": "Patient",
    "id": new_patient_id,
    "active": True,
    "name": [
        {
            "use": "official",
            "family": "Doe",
            "given": ["Jane"],
        }
    ],
    "gender": "female",
    "birthDate": "1990-06-15",
    "telecom": [
        {"system": "email", "value": "jane.doe@example.com", "use": "work"}
    ],
}

put_headers = {
    "Content-Type": "application/fhir+json",
    "Accept": "application/fhir+json",
}

put_response = requests.put(
    new_patient_url,
    headers=put_headers,
    json=new_patient,
)

print(f"PUT status code : {put_response.status_code}")
# 201 Created or 200 OK indicates success
print(f"New patient URI : {new_patient_url}")
print(f"Response body   : {put_response.text[:500]}")

### 3c. Verify the new record with a GET

Let's confirm that our newly created patient actually exists on the server by sending a `GET` request to the same URI we used in the `PUT`.

In [ ]:
verify_response = requests.get(new_patient_url, headers={"Accept": "application/fhir+json"})
print(f"GET status code : {verify_response.status_code}")
if verify_response.status_code == 200:
    print(json.dumps(verify_response.json(), indent=2))

### 3d. DELETE – Remove a resource

A `DELETE` request asks the server to *remove* a resource permanently. After a successful delete, subsequent `GET` requests to that URI will return a `410 Gone` or `404 Not Found` status code.

In [ ]:
# ── DELETE – remove the patient we just created ───────────────────────────────
delete_response = requests.delete(new_patient_url)

print(f"DELETE status code : {delete_response.status_code}")
# 200 OK or 204 No Content = resource was successfully deleted

In [ ]:
# Confirm the resource is gone
gone_response = requests.get(new_patient_url, headers={"Accept": "application/fhir+json"})
print(f"GET after DELETE status code : {gone_response.status_code}")
# Expect 410 (Gone) or 404 (Not Found)

---

## Summary

Congratulations! In this notebook you learned:

| Concept | What you did |
|---------|--------------|
| **REST API** | Sent HTTP requests to remote servers |
| **Resource URI** | Identified resources by their unique URL |
| **GET** | Retrieved a random joke and a FHIR Patient record |
| **JSON** | Parsed and worked with JSON data in Python |
| **POST** | Updated an existing patient's telecom information |
| **PUT** | Created a brand-new patient resource |
| **DELETE** | Removed a patient resource from the server |
| **FHIR** | Used a real healthcare interoperability standard |

### Next Steps

- Explore other FHIR resource types: `Observation`, `Condition`, `MedicationRequest`
- Try querying with search parameters, e.g., `/Patient?name=Smith`
- Look into OAuth 2.0 / SMART on FHIR for authenticated access to production systems
- Explore the official FHIR specification at https://hl7.org/fhir/R4/